# Exploración de los datos de entrada del autoencoder

Plantilla del change `ae-reconstruccion` (tarea 8.3). Corre en JupyterLab del Space
`sinca-dev`, con el rol del Space. **Nivel 1: sin TensorFlow.**

Explora la grilla horaria que consume el AE: cobertura y huecos por serie y canal,
ventanas que sobreviven según `L`, rangos tras la transformación, y el efecto de la
interpolación de ≤ 6 h y del recorte de auxiliares.

Convención: notebooks scratch, no fuente de verdad; se versionan sin outputs (`git add -f`).

In [ ]:
import sys, pathlib
# Raíz de importación del pipeline: el notebook vive en notebooks/ (o en la raíz del repo).
_here = pathlib.Path.cwd()
PIPELINE = next(p for p in (_here / "pipeline", _here.parent / "pipeline") if p.exists())
sys.path.insert(0, str(PIPELINE))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from analysis import load, plots_ae
from features.build import load_validated
from ae import data as aed
from ae import dmd as dmdm
from ae import metrics as aem

BUCKET_DATA = "sinca-data"
# Credenciales: el rol del Space (sinca-dev-role) las provee; no escribir claves aquí.

## 1. Cargar datos validados y armar la grilla

In [ ]:
validated = load_validated(BUCKET_DATA)  # S3 real; en local usar local_dir=...
grid = aed.build_grid(validated["sinca"], validated["open_meteo"], validated["feriados"])
print("horas:", len(grid.index), "| desde", grid.index.min(), "hasta", grid.index.max())

## 2. Cobertura y huecos por canal
Fracción de horas medidas (real), interpoladas (≤ 6 h) y huecos largos (> 6 h).

In [ ]:
rows = []
for col in grid.mp.columns:
    real = grid.mp_real[col].mean()
    long_gap = grid.mp[col].isna().mean()
    rows.append({"grupo": "mp", "canal": col, "real": real, "interpolado": 1 - real - long_gap, "hueco_largo": long_gap})
for col in grid.aux.columns:
    real = grid.aux_real[col].mean()
    long_gap = grid.aux[col].isna().mean()
    rows.append({"grupo": "aux", "canal": col, "real": real, "interpolado": 1 - real - long_gap, "hueco_largo": long_gap})
coverage = pd.DataFrame(rows)
display(coverage.round(3))

## 3. Ventanas que sobreviven según L (regla de descarte por huecos > 6 h)

In [ ]:
spans = aed.holdout_spans(grid.index)
survival = aed.survival_counts(grid, (24, 72, 168), spans)
display(survival)
fig = px.bar(survival, x="L", y="muestras", color="particion", barmode="group", title="Muestras por L y partición")
fig.show()

## 4. Rangos por canal tras la transformación (fit de normalización solo en train)

In [ ]:
train_mask = np.zeros(len(grid.index), dtype=bool)
train_mask[spans["train"][0]:spans["train"][1]] = True
norm = aed.fit_normalizer(grid, train_mask)
ng = aed.apply_normalizer(grid, norm)
ranges = aed.range_report(ng, real_rows=train_mask)
display(ranges.round(3))

## 5. Efecto de la interpolación ≤ 6 h y del recorte de auxiliares
Se compara la serie cruda con la llenada en un tramo con huecos.

In [ ]:
col = "pm25_nielol"
fig = go.Figure()
fig.add_trace(go.Scatter(x=grid.index, y=grid.mp[col], mode="lines", name="llenada (≤6 h)"))
fig.add_trace(go.Scatter(x=grid.index[grid.mp_real[col]], y=grid.mp[col][grid.mp_real[col]], mode="markers", name="real", marker=dict(size=3)))
fig.update_layout(title="pm25 Ñielol: valores reales y llenados", yaxis_title="µg/m³")
fig.show()